# Causal MAS Distillation - v3 run book

**Question this notebook answers:** is a multi-agent debate transcript a better
*training target* for a 1.5B student than a plain correct solution from the same
teacher, at a matched token budget?

- **Arm BASE** - untrained Qwen2.5-1.5B-Instruct. The sanity check.
- **Arm A** - fine-tune on one correct teacher solution (rejection sampling / STaR).
- **Arm B** - fine-tune on the whole debate transcript.
- **Arm C** - fine-tune on the debate's final solution only, critiques stripped.

A beats B and C -> the debate is not worth its cost as a data engine.
B beats A -> the *process* transfers, and that is the thesis.
C beats A but B does not -> the debate produces better *products*, not better *lessons*.

## Run order

| step | what | runtime | clock | cost |
|---|---|---|---|---|
| 1 | provider audit | CPU | 2 min | ~0 |
| 2 | reuse the existing probe | CPU | 1 min | 0 |
| 3 | generate traces (stratified) | CPU | 1-1.5 h | ~$10 |
| 4 | **validation gate** | CPU | 5 min | 0 |
| 5 | build Arm A from the probe cache | CPU | 3 min | 0 |
| 6 | build A/B/C datasets | CPU | 2 min | 0 |
| 7 | **dry-run gate** | CPU | 2 min | 0 |
| 8 | train 4 arms x 3 seeds | **T4** | 3-5 h | 0 |
| 9 | evaluate | **T4** | 40 min | 0 |

Steps 1-7 need no GPU. Start them on a CPU runtime, then switch to T4 for 8-9.

## Two decisions already made, do not re-litigate them

1. **The probe is not re-run.** `data/probed_all.json` was generated before we
   discovered that GeneralCompute ignores `max_tokens`, which means it ran
   *uncapped*, which is exactly what you want when measuring a true pass rate.
   It is the one artefact in this project that was never contaminated.
   Re-running it with a cap that now actually binds would truncate ~50% of
   solutions before `\boxed{}` and collapse the band distribution.
2. **The critic is `gpt-oss-120b`, not the solver.** One model playing both
   solver and critic is self-critique, and it scored a dispute rate of 0.080
   with recall 0.100. That is the finding of Huang et al., ICLR 2024, not a
   contribution.

---
## Step 0 - environment

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

# v3 outputs are isolated. The one thing we deliberately READ from the old run
# is the probe cache, because it IS Arm A.
ROOT      = '/content/drive/MyDrive/cmd'
V3        = f'{ROOT}/v3'
PROBE_CACHE = f'{ROOT}/cache_probe.jsonl'      # <-- reused, read-only
TRACE_CACHE = f'{V3}/cache_traces.jsonl'        # <-- fresh
CKPT_ROOT = f'{V3}/ckpt'

for d in (V3, CKPT_ROOT):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

assert os.path.exists(PROBE_CACHE), (
    f'missing {PROBE_CACHE}. Arm A is reconstructed from this file; without it '
    'you would have to re-probe (~$4, 1 h).')
print('probe cache :', os.path.getsize(PROBE_CACHE) // 1024, 'KB')
print('v3 dir      :', V3)

Mounted at /content/drive
probe cache : 81386 KB
v3 dir      : /content/drive/MyDrive/cmd/v3


In [2]:
%cd /content
![ -d causal-mas-distill ] || git clone https://github.com/Arshia-HZ/causal-mas-distill.git
%cd /content/causal-mas-distill
!git pull --ff-only
!pip -q install -r requirements-api.txt

# Fail loudly now rather than in step 3.
import pathlib
need = ['scripts/00j_provider_audit.py', 'scripts/01b_generate_traces.py',
        'scripts/00f_check_traces.py', 'scripts/00g_diagnose_signal.py',
        'scripts/10_build_arm_a.py', 'scripts/11_build_abc_datasets.py',
        'scripts/04b_train_ab.py', 'scripts/12_eval_abc.py',
        'src/backends/multikey.py', 'src/debate/harness_debate.py',
        'src/debate/prompts.py']
missing = [p for p in need if not pathlib.Path(p).exists()]
print('MISSING:', missing if missing else 'none')
assert not missing, 'copy these from the handoff patches/ folder first'

/content
Cloning into 'causal-mas-distill'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 325 (delta 138), reused 284 (delta 97), pack-reused 0 (from 0)
Receiving objects: 100% (325/325), 233.69 KiB | 1.69 MiB/s, done.
Resolving deltas: 100% (138/138), done.
/content/causal-mas-distill
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 3.5 MB/s eta 0:00:00
MISSING: none


### API keys

Put the keys in **Colab Secrets** (key icon in the left sidebar), never in a cell.
Create one secret named `GC_API_KEYS` whose value is your four keys separated by
commas, no spaces:

```
k_aaa,k_bbb,k_ccc,k_ddd
```

This notebook is committed to a public repo. Colab saves cell *output* into the
`.ipynb`, so a key echoed once is a key leaked forever.

In [3]:
import os
from google.colab import userdata

os.environ['GC_API_KEYS'] = userdata.get('GC_API_KEYS')
os.environ['GC_BASE_URL'] = 'https://api.generalcompute.com/v1'

# Heterogeneous agents. The critic MUST differ from the solver.
os.environ['SOLVER_MODEL']   = 'deepseek-v3.2'
os.environ['CRITIC_MODEL']   = 'gpt-oss-120b'
os.environ['VERIFIER_MODEL'] = 'deepseek-v3.2'

n = len([k for k in os.environ['GC_API_KEYS'].split(',') if k.strip()])
print(f'{n} key(s) loaded')          # prints the COUNT, never the keys
assert n >= 1

4 key(s) loaded


---
## Step 1 - provider audit (2 min)

One unresolved contradiction: the dashboard advertises 32k context for
`deepseek-v3.2`, but a real run returned
`Input is 8357 tokens but this model only supports 8192`.

We also now know GeneralCompute reads `max_completion_tokens` and silently
ignores `max_tokens`. `multikey.py` sends the correct one and falls back
automatically, but this cell **proves** the cap binds before you spend money.

In [8]:
!python scripts/00j_provider_audit.py --mode limits \
  --models $SOLVER_MODEL $CRITIC_MODEL \
  --out results/provider_limits.json

=== CONTEXT AND max_tokens ===
model                      real_ctx     max_tok_ok note
[multikey] 4 key(s), 4 concurrent each = 16 total
deepseek-v3.2             57788 tok     NO (400 w) Error code: 400 - {'error': {'message': "This model's maximum context length is 32768 toke
[multikey] calls per key : [0, 0, 0, 0]
[multikey] 429s per key  : [0, 0, 0, 0]
[multikey] 4 key(s), 4 concurrent each = 16 total
gpt-oss-120b          >199980 chars     NO (400 w) no overflow at probe size; raise --probe-chars
[multikey] calls per key : [0, 0, 0, 0]
[multikey] 429s per key  : [0, 0, 0, 0]

wrote results/provider_limits.json


Read two numbers out of that output:

- **effective context** per model -> the ceiling on transcript length.
- **cap honoured?** must say YES. If a request for 64 tokens returns 900, the
  cap is still being ignored and every length control downstream is fiction.

---
## Step 2 - reuse the existing probe (1 min, free)

No generation here. This cell just re-reads what you already paid for and
restates the bands, so the numbers in your thesis come from a file you can
point at rather than from memory.

In [4]:
import json, collections

P = json.load(open('data/probed_all.json'))
print('probed problems:', len(P))

band = collections.Counter()
for r in P:
    p = float(r.get('pass_rate', 0.0))
    band['floor  (p=0)'   if p == 0.0 else
         'ceiling (p=1)'  if p >= 1.0 else
         'headroom(0<p<1)'] += 1
for k in ('floor  (p=0)', 'headroom(0<p<1)', 'ceiling (p=1)'):
    print(f'  {k:18s} {band[k]:4d}  {100*band[k]/len(P):5.1f}%')

nonceiling = sum(1 for r in P if float(r.get('pass_rate', 0)) < 1.0)
print(f'\nstep 3 will spend 3 seeds on {nonceiling} problems '
      f'and 1 seed on {len(P)-nonceiling}')

assert len(P) >= 700, 'expected ~785 probed problems'

probed problems: 785
  floor  (p=0)         22    2.8%
  headroom(0<p<1)     134   17.1%
  ceiling (p=1)       629   80.1%

step 3 will spend 3 seeds on 156 problems and 1 seed on 629


**Why the bands matter, and what they do NOT mean.** `pass_rate` measures
difficulty *for the teacher*, at k=32. A problem at `p=1.0` is not easy for a
1.5B student, so ceiling problems are still useful training data. The bands are
used for two things only:

1. deciding how many debate seeds to spend per problem (below), and
2. stratifying the final evaluation, since an effect that only appears on
   problems the teacher already solves perfectly is a different claim.

---
## Step 3 - generate debate traces (1-1.5 h, ~$10)

**Stratified on purpose.** `--n-solutions` is not a variance-reduction knob, it
is a *coverage* knob. If every trace for a problem ends up wrong, that problem
is dropped from Arm B entirely, and then dropped from A and C too because
`11_build_abc_datasets.py` intersects the three arms. Measured on the previous
run:

| n_solutions | problems with >= 1 correct trace |
|---|---|
| 1 | 78.7% |
| 2 | 85.2% |
| 3 | **92.5%** |

The ~21% lost at n=1 are not spread evenly. They are almost all hard problems,
which is exactly where the hypothesis lives. So: 1 seed on the ceiling band
where the solver is right anyway, 3 seeds where coverage actually fails. Half
the cost of 3-everywhere, same coverage where it counts.

`--max-tokens 4096`: with the cap now genuinely binding, 1024 would truncate
~51% of solver messages before `\boxed{}` and 768 would truncate ~68%.

In [6]:
# 3a. CEILING band (p = 1.0): one seed each.
!python scripts/01b_generate_traces.py \
    --problems data/probed_all.json \
    --output data/traces_v3.jsonl \
    --cache-path "/content/drive/MyDrive/cmd/v3/cache_traces.jsonl" \
    --api-url "$GC_BASE_URL" --api-keys-env GC_API_KEYS \
    --solver-model "$SOLVER_MODEL" \
    --critic-model "$CRITIC_MODEL" \
    --verifier-model "$VERIFIER_MODEL" \
    --critic-persona adversarial \
    --min-p 1.0 \
    --n-solutions 1 \
    --max-rounds 3 --max-tokens 4096 \
    --concurrency-per-key 8 --problem-concurrency 24 \
    --chunk 25 --resume

problems loaded : 629
roles           : {'solver': 'deepseek-v3.2', 'critic': 'gpt-oss-120b', 'verifier': 'deepseek-v3.2'}
[multikey] 4 key(s), 8 concurrent each = 32 total
[multikey] 4 key(s), 8 concurrent each = 32 total
[25/629 problems] 25 traces | 201s elapsed | ETA 81 min
    critic dispute rate so far: 0.000
[50/629 problems] 50 traces | 566s elapsed | ETA 109 min
    critic dispute rate so far: 0.000
[75/629 problems] 75 traces | 1038s elapsed | ETA 128 min
    critic dispute rate so far: 0.040
[100/629 problems] 100 traces | 1239s elapsed | ETA 109 min
    critic dispute rate so far: 0.000
[125/629 problems] 125 traces | 1564s elapsed | ETA 105 min
    critic dispute rate so far: 0.000
[150/629 problems] 150 traces | 2179s elapsed | ETA 116 min
    critic dispute rate so far: 0.000
[175/629 problems] 175 traces | 2848s elapsed | ETA 123 min
    critic dispute rate so far: 0.020
[200/629 problems] 200 traces | 3184s elapsed | ETA 114 min
    critic dispute rate so far: 0.000
[2

In [7]:
# 3b. NON-CEILING band (p < 1.0): three seeds each. Appends to the same file.
!python scripts/01b_generate_traces.py \
    --problems data/probed_all.json \
    --output data/traces_v3.jsonl \
    --cache-path "/content/drive/MyDrive/cmd/v3/cache_traces.jsonl" \
    --api-url "$GC_BASE_URL" --api-keys-env GC_API_KEYS \
    --solver-model "$SOLVER_MODEL" \
    --critic-model "$CRITIC_MODEL" \
    --verifier-model "$VERIFIER_MODEL" \
    --critic-persona adversarial \
    --max-p 0.999 \
    --n-solutions 3 \
    --max-rounds 3 --max-tokens 4096 \
    --concurrency-per-key 8 --problem-concurrency 24 \
    --chunk 25 --resume

problems loaded : 156
already in output: 629 -> 156 remaining
roles           : {'solver': 'deepseek-v3.2', 'critic': 'gpt-oss-120b', 'verifier': 'deepseek-v3.2'}
[multikey] 4 key(s), 8 concurrent each = 32 total
[multikey] 4 key(s), 8 concurrent each = 32 total
[25/156 problems] 75 traces | 685s elapsed | ETA 60 min
    critic dispute rate so far: 0.073
[50/156 problems] 150 traces | 1484s elapsed | ETA 52 min
    critic dispute rate so far: 0.082
[75/156 problems] 225 traces | 2406s elapsed | ETA 43 min
    critic dispute rate so far: 0.128
[100/156 problems] 300 traces | 3572s elapsed | ETA 33 min
    critic dispute rate so far: 0.209
[125/156 problems] 375 traces | 4458s elapsed | ETA 18 min
    critic dispute rate so far: 0.094
[150/156 problems] 450 traces | 5125s elapsed | ETA 3 min
    critic dispute rate so far: 0.101
[156/156 problems] 468 traces | 5312s elapsed | ETA 0 min
    critic dispute rate so far: 0.056
[multikey] calls per key : [234, 234, 234, 234]
[multikey] 429s p

In [8]:
# Back up the traces before anything can overwrite them. This file cost real money.
!cp data/traces_v3.jsonl "/content/drive/MyDrive/cmd/v3/traces_v3.jsonl"
!ls -la "/content/drive/MyDrive/cmd/v3"

total 18302
-rw------- 1 root root 1746017 Aug 13 14:34 cache_probe.jsonl
-rw------- 1 root root 8039757 Aug 13 20:30 cache_traces.jsonl
drwx------ 2 root root    4096 Aug 13 13:52 ckpt
-rw------- 1 root root 8949981 Aug 13 20:32 traces_v3.jsonl


---
## Step 4 - VALIDATION GATE (5 min, free)

**Do not skip this and do not proceed on a bad result.** Everything after this
point costs GPU hours, and on 13 Aug five hours were spent training on data
that contained no signal.

In [14]:
!python scripts/00g_diagnose_signal.py \
    --traces data/traces_v3.jsonl --probed data/probed_all.json

SIGNAL DIAGNOSIS v3  (1097 traces)
grader : eval.grade.is_correct
cap    : 4096 tokens (heuristic char cap = 12697)
COMPLETENESS
  solver msgs with no answer  : 111/3291 (3.4%)   [gate: <= 5%]
  verifier msgs with no answer: 0/1097 (0.0%)   [gate: <= 1%]
  messages near 4096-token cap   : 23/6572 (0.3%)
  short traces (<6 msgs)      : 9   (resilient-retry degradation; builder drops them)
------------------------------------------------------------------
SEED INDEPENDENCE over 156 multi-seed problems
  problems with identical round-1 seeds: 0   [gate: 0]
------------------------------------------------------------------
STRUCTURE
  messages per trace : {4: 1, 5: 8, 6: 1088}
  roles              : {'solver': 3291, 'critic': 2184, 'verifier': 1097}
  seeds per problem  : {1: 629, 3: 156}
  per-pid acc spread : {0.0: 24, 0.333: 19, 0.667: 21, 1.0: 721}
------------------------------------------------------------------
DOES THE DEBATE MOVE THE ANSWER?  (re-graded, one grader)
  traces score

### Pass / fail

| metric | v2 (broken) | v3 must reach | if it fails |
|---|---|---|---|
| critic dispute rate | 0.080 | **>= 0.25** | prompt problem, 20 min fix, not a model problem |
| critic recall on genuinely wrong answers | 0.100 | **>= 0.30** | the critic is still decorative |
| problems with >= 1 correct trace | 123 | **>= 600** | coverage collapse, Arm B trains on nothing |
| solver messages with no boxed answer | 9.2% | **<= 5%** | raise `--max-tokens` |
| `VERDICT:` lines that fail to parse | n/a | **<= 5%** | critic truncated, raise its cap |

The dispute rate is the single number that says whether swapping the critic to
`gpt-oss-120b` did anything. If it is still under 0.15, stop here: you have a
prompt bug, and burning 5 GPU hours will not reveal it.

If dispute rate lands between 0.15 and 0.25, continue, but write the number in
the paper. A weak critic is a legitimate finding as long as you measured it
instead of assuming it.

---
## Step 5 - build Arm A from the probe cache (3 min, free)

Arm A is rejection sampling: keep the correct solutions the teacher already
produced during the probe. No new API calls.

**`--max-tokens 768` is not a typo and must not be "fixed".** The cache key is
`sha256({messages, n, temperature, max_tokens, model, nonce})`. The old probe
ran with 768 in that field, so 768 is what makes the keys hit. Change it to
1024 and recovery silently drops to 0% and Arm A comes out empty.

In [ ]:
!python scripts/10_build_arm_a.py \
    --cache-path "$PROBE_CACHE" \
    --problems data/probed_all.json \
    --model "$SOLVER_MODEL" \
    --n-probe 32 --max-tokens 768 --temperature 0.7 \
    --out data/arm_a_pool.jsonl

`recovery rate` must be at or near **100%**. Anything lower means a cache-key
mismatch: check `--max-tokens`, `--model`, `--temperature`, and that
`10_build_arm_a.py` still uses the frozen `LEGACY_SOLVE_PROMPT` string rather
than importing today's prompt from `src/debate/prompts.py`. That prompt text is
part of the key.

---
## Step 6 - build the A/B/C datasets (2 min, free)

This is where the previous run went wrong in two ways at once, both now guarded:

- **coverage**: Arm A got 67 problems, Arm B got 22. Any measured difference was
  a difference in problem sets, not in data format. The builder now intersects
  all three arms and refuses to continue if coverage is lopsided.
- **tokens**: debate transcripts are longer, so "same number of examples" means
  "more tokens for B". The builder matches *completion tokens* to within 1%.

The `_eqp` variant (equal problems per arm, one example each) is the **primary**
result. The `_tok` variant is the secondary check.

In [ ]:
!python scripts/11_build_abc_datasets.py \
    --arm-a-pool data/arm_a_pool.jsonl \
    --traces data/traces_v3.jsonl \
    --probed data/probed_all.json \
    --budget-tokens 400000 \
    --clip-critic-words 200 \
    --max-rounds-render 2 \
    --max-completion-tokens 3584 \
    --eval-mod 5 --eval-extra 200 \
    --seed 0 --outdir data

Check before moving on:

- `INTERSECTION` should now be in the many hundreds, not 123.
- the token gap between arms must be under 1%.
- the eval set should be **>= 150 problems**. At n=32 the confidence interval is
  +/-9 points, which cannot resolve the ~2-point effect we are looking for.

---
## Step 7 - DRY-RUN GATE (2 min, free, no GPU)

On 13 Aug the trainer reported `loss 2.012e-07` and finished in 19 seconds. The
cause: `format_example()` in the old `src/distill/sft.py` looked for keys named
`input` / `output`, found neither, and rewrote every row to the constant string
`"\nAnswer: "`. Five tokens per example. Both checkpoints were noise.

`04b_train_ab.py --dry-run` tokenises the dataset, prints the actual supervised
span, and **exits non-zero** if fewer than 80% of examples survive. It imports
no torch. Run it on every dataset, every time.

In [ ]:
import subprocess, sys

bad = []
for arm in ('a', 'b', 'c'):
    ds = f'data/sft_arm_{arm}_eqp.jsonl'
    print(f'\n================ dry run: {ds} ================')
    r = subprocess.run([sys.executable, 'scripts/04b_train_ab.py',
                        '--dataset', ds, '--max-seq-length', '4096',
                        '--dry-run'])
    if r.returncode != 0:
        bad.append(ds)

assert not bad, f'FIX THESE BEFORE TRAINING: {bad}'
print('\nall datasets tokenise correctly')

Read the printed supervised span with your own eyes. It must be the teacher's
reasoning text. If it looks like a prompt, or is 5 tokens long, or is empty, the
bug is back.

---
## Step 8 - train (3-5 h)

**Switch the runtime to T4 now:** Runtime -> Change runtime type -> T4 GPU.
Then re-run the Step 0 cells (mount, cd, keys) before this one.

Four arms x three seeds. Three seeds is not optional: seed-to-seed spread on a
1.5B LoRA is the same order of magnitude as the effect being measured, so a
single seed is not a result.

`bf16` is disabled automatically on a T4 (sm_75 has no bf16). `max_seq_length`
stays at 4096: the 8192 x 151936 logits tensor is ~2.5 GB in fp16 on top of
3.1 GB of weights, which will OOM.

In [ ]:
!pip -q install -r requirements-train.txt
import torch
print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())

In [ ]:
import os, subprocess, sys, time

V3 = '/content/drive/MyDrive/cmd/v3'
VARIANT = 'eqp'          # primary result. Re-run with 'tok' afterwards.
t0 = time.time()

for arm in ('a', 'b', 'c'):
    for seed in (0, 1, 2):
        out = f'{V3}/ckpt/arm_{arm}_{VARIANT}/seed{seed}'
        if os.path.exists(f'{out}/final'):
            print('skip (done):', out); continue
        print(f'\n===== arm {arm} / seed {seed} =====  t+{(time.time()-t0)/60:.0f} min')
        r = subprocess.run([sys.executable, 'scripts/04b_train_ab.py',
                            '--dataset', f'data/sft_arm_{arm}_{VARIANT}.jsonl',
                            '--output-dir', out,
                            '--max-seq-length', '4096',
                            '--epochs', '3', '--seed', str(seed)])
        assert r.returncode == 0, f'arm {arm} seed {seed} failed'

print(f'\ndone in {(time.time()-t0)/60:.0f} min')

---
## Step 9 - evaluate (40 min)

Arm BASE is the untrained model. It is listed first on purpose: **if A and B are
both below BASE, the training pipeline is broken and the hypothesis was never
tested.** That is the check that was missing on 13 Aug.

Greedy decoding, no sampling, so the only variation across seeds is the
checkpoint itself.

In [ ]:
V3 = '/content/drive/MyDrive/cmd/v3'
VARIANT = 'eqp'

!python scripts/12_eval_abc.py \
    --eval data/eval_problems.json \
    --probed data/probed_all.json \
    --arm-base base \
    --arm-a {V3}/ckpt/arm_a_{VARIANT}/seed0/final {V3}/ckpt/arm_a_{VARIANT}/seed1/final {V3}/ckpt/arm_a_{VARIANT}/seed2/final \
    --arm-b {V3}/ckpt/arm_b_{VARIANT}/seed0/final {V3}/ckpt/arm_b_{VARIANT}/seed1/final {V3}/ckpt/arm_b_{VARIANT}/seed2/final \
    --arm-c {V3}/ckpt/arm_c_{VARIANT}/seed0/final {V3}/ckpt/arm_c_{VARIANT}/seed1/final {V3}/ckpt/arm_c_{VARIANT}/seed2/final \
    --max-new-tokens 1024 \
    --out results/abc_eval_{VARIANT}.json

### How to read the output

Read the rows in this order.

1. **BASE.** If A, B and C are all at or below BASE, stop. Fine-tuning made the
   model worse, which is a pipeline failure, not a finding.
2. **A vs BASE.** Arm A is rejection-sampling fine-tuning, a published method.
   It should be clearly above BASE. If it is not, the training recipe is wrong
   and nothing else in the table can be interpreted.
3. **B - A**, with its paired-bootstrap interval. This is the thesis.
4. **C - A** and **B - C**. These separate the two ways the debate could help:
   better final answers (C) versus a transferable critique-and-revise process
   (B). If C beats A but B does not, the debate is a better *generator*, not a
   better *teacher* - which is a perfectly publishable result and a much
   cheaper pipeline.
5. **Strata.** An effect that appears only on teacher-ceiling problems means
   something different from one that appears on hard problems. Report both.

A null with a tight interval is a result. A null with a +/-9 point interval is
an underpowered experiment; if you see that, enlarge the eval set rather than
writing it up.

---
## After this

This notebook tests the **precondition**: is debate text a better training
target at all? Only if B or C wins does the original thesis question become
measurable, namely *which messages* carry the value:

```bash
# 1. placebo FIRST. Must print exactly 0.0. If not, the estimator is leaking.
python -c "from src.counterfactual.crn import placebo_check; print(placebo_check())"

# 2. then the real thing
python scripts/02_counterfactual_replay.py \
    --traces data/traces_v3.jsonl --estimand trace_utilities_total_crn --k 32
```

If B and C both come back null with tight intervals, do not force it. "A matched
budget of rejection samples beats debate transcripts as distillation data, and
here is the coverage analysis explaining why" is a defensible thesis and an
honest paper. It needs a real baseline, matched budgets, three seeds and a
confidence interval - all of which this notebook produces.